In [1]:
### 1. Download Dataset

!gdown --id '1u1OjLJLf2kPRJicGSgnBwiWPOtl6eaN6' --output hw7_data.zip
!unzip -o hw7_data.zip
# For this HW, K80 < P4 < T4 < P100 <= T4(fp16) < V100
!nvidia-smi

/usr/local/lib/python3.12/dist-packages/gdown/__main__.py:140: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  warnings.warn(
Downloading...
From: https://drive.google.com/uc?id=1u1OjLJLf2kPRJicGSgnBwiWPOtl6eaN6
To: /content/hw7_data.zip
100% 11.5M/11.5M [00:00<00:00, 55.6MB/s]
Archive:  hw7_data.zip
  inflating: hw7_dev.json            
  inflating: hw7_test.json           
  inflating: hw7_train.json          
Thu Nov 13 14:31:24 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M.

In [2]:
### 2. Install transformers
# You are allowed to change version of transformers or use other toolkits
# I choose this： https://huggingface.co/deepset/roberta-base-squad2
# !pip install transformers==4.5.0
!pip install transformers
!pip install accelerate

In [ ]:
### 3.1 Import Packages
import json
import numpy as np
import random
import torch
from torch.utils.data import DataLoader, Dataset
from transformers import BertForQuestionAnswering, BertTokenizerFast
from torch.optim import AdamW
from tqdm.auto import tqdm

device = "cuda" if torch.cuda.is_available() else "cpu"


# Fix random seed for reproducibility (為了保證可複現性，請固定隨機種子。)
def same_seeds(seed):
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    random.seed(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True


same_seeds(0)

In [ ]:
### 3.2
# Change "fp16_training" to True to support automatic mixed precision training (fp16)
fp16_training = False

if fp16_training:
    !pip install accelerate==0.2.0
    from accelerate import Accelerator

    accelerator = Accelerator(fp16=True)
    device = accelerator.device

# Documentation for the toolkit:  https://huggingface.co/docs/accelerate/

In [1]:
### 4. Load Model and Tokenizer

model = BertForQuestionAnswering.from_pretrained("bert-base-chinese").to(device)
tokenizer = BertTokenizerFast.from_pretrained("bert-base-chinese")

# You can safely ignore the warning message (it pops up because new prediction heads for QA are initialized randomly)
# (您可以忽略此警告訊息（它彈出是因為 QA 的新預測頭是隨機初始化的）。)

NameError: name 'BertForQuestionAnswering' is not defined

In [ ]:
### 5. Read Data
def read_data(file):
    with open(file, "r", encoding="utf-8") as reader:
        data = json.load(reader)
    return data["questions"], data["paragraphs"]


train_questions, train_paragraphs = read_data("hw7_train.json")
dev_questions, dev_paragraphs = read_data("hw7_dev.json")
test_questions, test_paragraphs = read_data("hw7_test.json")

In [ ]:
### 6. Tokenize Data
# Tokenize questions and paragraphs separately
# 「add_special_tokens」 is set to False since special tokens will be added when tokenized questions and paragraphs are combined in datset __getitem__
# (「add_special_tokens」 設定為 False，因為在資料集 __getitem__ 中合併分詞後的問題和段落時會新增特殊標記。)

train_questions_tokenized = tokenizer(
    [train_question["question_text"] for train_question in train_questions],
    add_special_tokens=False,
)
dev_questions_tokenized = tokenizer(
    [dev_question["question_text"] for dev_question in dev_questions],
    add_special_tokens=False,
)
test_questions_tokenized = tokenizer(
    [test_question["question_text"] for test_question in test_questions],
    add_special_tokens=False,
)

train_paragraphs_tokenized = tokenizer(train_paragraphs, add_special_tokens=False)
dev_paragraphs_tokenized = tokenizer(dev_paragraphs, add_special_tokens=False)
test_paragraphs_tokenized = tokenizer(test_paragraphs, add_special_tokens=False)

# You can safely ignore the warning message as tokenized sequences will be futher processed in datset __getitem__ before passing to model
# (您可以忽略此警告訊息，因為分詞後的序列在傳遞給模型之前會在資料集__getitem__中進一步處理。)

Token indices sequence length is longer than the specified maximum sequence length for this model (570 > 512). Running this sequence through the model will result in indexing errors


In [ ]:
### 7. Dataset and Dataloader
class QA_Dataset(Dataset):
    def __init__(self, split, questions, tokenized_questions, tokenized_paragraphs):
        self.split = split
        self.questions = questions
        self.tokenized_questions = tokenized_questions
        self.tokenized_paragraphs = tokenized_paragraphs
        self.max_question_len = 40
        self.max_paragraph_len = 150
        ##### TODO: Change value of doc_stride #####
        self.doc_stride = 80  # Boss 建議值：64~96
        ##### TODO: Preprocessing #####
        self.max_seq_len = 1 + self.max_question_len + 1 + self.max_paragraph_len + 1

    def __len__(self):
        return len(self.questions)

    def __getitem__(self, idx):
        question = self.questions[idx]
        tokenized_question = self.tokenized_questions[idx]
        tokenized_paragraph = self.tokenized_paragraphs[question["paragraph_id"]]

        ##### TODO: Preprocessing #####
        # 原本答案永遠置中 → 加入隨機偏移
        if self.split == "train":
            # 找出答案在段落中的 token 位置
            answer_start_token = question["answer_start_token"]
            answer_end_token = question["answer_end_token"]

            # 計算答案長度
            answer_len = answer_end_token - answer_start_token + 1

            # 隨機決定窗口起始位置（讓答案不會永遠在正中間）
            max_start = len(tokenized_paragraph) - self.max_paragraph_len
            if max_start > 0:
                # 讓答案有機會出現在窗口的不同位置
                random_offset = random.randint(0, min(self.doc_stride, max_start))
                paragraph_start = max(
                    0,
                    min(
                        answer_start_token
                        - self.max_paragraph_len // 2
                        + random_offset,
                        max_start,
                    ),
                )
            else:
                paragraph_start = 0
        else:
            paragraph_start = 0

        # 建立 input
        input_ids_question = (
            [101] + tokenized_question.ids[: self.max_question_len] + [102]
        )
        input_ids_paragraph = tokenized_paragraph.ids[
            paragraph_start : paragraph_start + self.max_paragraph_len
        ] + [102]

        # padding
        padding_len = (
            self.max_seq_len - len(input_ids_question) - len(input_ids_paragraph)
        )
        input_ids = input_ids_question + input_ids_paragraph + [0] * padding_len
        token_type_ids = (
            [0] * len(input_ids_question)
            + [1] * len(input_ids_paragraph)
            + [0] * padding_len
        )
        attention_mask = [1] * (len(input_ids_question) + len(input_ids_paragraph)) + [
            0
        ] * padding_len

        if self.split == "train":
            answer_start = len(input_ids_question) + (
                answer_start_token - paragraph_start
            )
            answer_end = len(input_ids_question) + (answer_end_token - paragraph_start)
            # 確保答案在合法範圍內
            if (
                answer_start < len(input_ids_question)
                or answer_end >= self.max_seq_len - 1
            ):
                answer_start = 0
                answer_end = 0
        else:
            answer_start = answer_end = 0

        output = {
            "input_ids": input_ids,
            "token_type_ids": token_type_ids,
            "attention_mask": attention_mask,
            "start_positions": answer_start,
            "end_positions": answer_end,
        }
        return output


train_set = QA_Dataset(
    "train", train_questions, train_questions_tokenized, train_paragraphs_tokenized
)
dev_set = QA_Dataset(
    "dev", dev_questions, dev_questions_tokenized, dev_paragraphs_tokenized
)
test_set = QA_Dataset(
    "test", test_questions, test_questions_tokenized, test_paragraphs_tokenized
)

train_batch_size = 16

# Note: Do NOT change batch size of dev_loader / test_loader !
# (請勿更改 dev_loader / test_loader 的批次大小)
# Although batch size=1, it is actually a batch consisting of several windows from the same QA pair
# (雖然批次大小為 1，但實際上它是一個包含來自相同 QA 對的多個視窗的批次。)
train_loader = DataLoader(
    train_set, batch_size=train_batch_size, shuffle=True, pin_memory=True
)
dev_loader = DataLoader(dev_set, batch_size=1, shuffle=False, pin_memory=True)
test_loader = DataLoader(test_set, batch_size=1, shuffle=False, pin_memory=True)

In [ ]:
### 8. Function for Evaluation
def evaluate(data, output, max_answer_len=30):
    ##### TODO: Postprocessing #####
    answer = ""
    max_prob = float("-inf")
    num_of_windows = data[0].shape[1]

    # 遍歷所有窗口
    for k in range(num_of_windows):
        # 取得當前窗口的預測
        start_prob = output.start_logits[0][k].cpu().numpy()
        end_prob = output.end_logits[0][k].cpu().numpy()
        token_type_ids = data[1][0][k].cpu().numpy()
        attention_mask = data[2][0][k].cpu().numpy()

        # 找出 paragraph 的有效範圍（token_type_ids == 1 且 attention_mask == 1）
        paragraph_indices = np.where((token_type_ids == 1) & (attention_mask == 1))[0]

        if len(paragraph_indices) == 0:
            continue

        paragraph_start = paragraph_indices[0]
        paragraph_end = paragraph_indices[-1]

        # 只在合法範圍內搜尋
        for i in range(paragraph_start, min(paragraph_end + 1, len(start_prob))):
            for j in range(i, min(paragraph_end + 1, len(end_prob))):
                if j - i + 1 > max_answer_len:
                    break
                prob = start_prob[i] + end_prob[j]
                if prob > max_prob:
                    max_prob = prob
                    # 轉回文字
                    answer = tokenizer.decode(data[0][0][k][i : j + 1])

    return answer

In [ ]:
### 9. Training
from transformers import get_linear_schedule_with_warmup

num_epoch = 2  # 建議 2~3
doc_stride = 80  # 已在上方 Dataset 中設定
max_paragraph_len = 150
max_answer_len = 30
learning_rate = 3e-5
fp16_training = True  # 可開啟加速
validation = True
logging_step = 100
learning_rate = 3e-5  # Boss 建議值
optimizer = AdamW(model.parameters(), lr=learning_rate)

# 使用 linear warmup + decay
total_steps = len(train_loader) * num_epoch
warmup_steps = int(total_steps * 0.1)  # 10% warmup

scheduler = get_linear_schedule_with_warmup(
    optimizer, num_warmup_steps=warmup_steps, num_training_steps=total_steps
)

if fp16_training:
    model, optimizer, train_loader = accelerator.prepare(model, optimizer, train_loader)

model.train()

print("Start Training ...")

for epoch in range(num_epoch):
    step = 1
    train_loss = train_acc = 0

    for data in tqdm(train_loader):
        # Load all data into GPU
        data = [i.to(device) for i in data]

        # Model inputs: input_ids, token_type_ids, attention_mask, start_positions, end_positions (Note: only "input_ids" is mandatory)
        # Model outputs: start_logits, end_logits, loss (return when start_positions/end_positions are provided)
        output = model(
            input_ids=data[0],
            token_type_ids=data[1],
            attention_mask=data[2],
            start_positions=data[3],
            end_positions=data[4],
        )

        # Choose the most probable start position / end position
        start_index = torch.argmax(output.start_logits, dim=1)
        end_index = torch.argmax(output.end_logits, dim=1)

        # Prediction is correct only if both start_index and end_index are correct
        train_acc += ((start_index == data[3]) & (end_index == data[4])).float().mean()
        train_loss += output.loss

        if fp16_training:
            accelerator.backward(output.loss)
        else:
            output.loss.backward()

        optimizer.step()
        scheduler.step()
        optimizer.zero_grad()
        step += 1

        ##### TODO: Apply linear learning rate decay #####

        # Print training loss and accuracy over past logging step
        if step % logging_step == 0:
            print(
                f"Epoch {epoch + 1} | Step {step} | loss = {train_loss.item() / logging_step:.3f}, acc = {train_acc / logging_step:.3f}"
            )
            train_loss = train_acc = 0

    if validation:
        print("Evaluating Dev Set ...")
        model.eval()
        with torch.no_grad():
            dev_acc = 0
            for i, data in enumerate(tqdm(dev_loader)):
                output = model(
                    input_ids=data[0].squeeze(dim=0).to(device),
                    token_type_ids=data[1].squeeze(dim=0).to(device),
                    attention_mask=data[2].squeeze(dim=0).to(device),
                )
                # prediction is correct only if answer text exactly matches
                # (只有當答案文字與預測完全匹配時，預測才是正確的。)
                dev_acc += evaluate(data, output) == dev_questions[i]["answer_text"]
            print(
                f"Validation | Epoch {epoch + 1} | acc = {dev_acc / len(dev_loader):.3f}"
            )
        model.train()

# Save a model and its configuration file to the directory 「saved_model」
# i.e. there are two files under the direcory 「saved_model」: 「pytorch_model.bin」 and 「config.json」
# Saved model can be re-loaded using 「model = BertForQuestionAnswering.from_pretrained("saved_model")」
print("Saving Model ...")
model_save_dir = "saved_model"
model.save_pretrained(model_save_dir)

Start Training ...


  0%|          | 0/1621 [00:00<?, ?it/s]

Epoch 1 | Step 100 | loss = nan, acc = 0.000
Epoch 1 | Step 200 | loss = nan, acc = 0.000
Epoch 1 | Step 300 | loss = nan, acc = 0.000
Epoch 1 | Step 400 | loss = nan, acc = 0.000
Epoch 1 | Step 500 | loss = nan, acc = 0.000
Epoch 1 | Step 600 | loss = nan, acc = 0.000
Epoch 1 | Step 700 | loss = nan, acc = 0.000
Epoch 1 | Step 800 | loss = nan, acc = 0.000
Epoch 1 | Step 900 | loss = nan, acc = 0.000
Epoch 1 | Step 1000 | loss = nan, acc = 0.000
Epoch 1 | Step 1100 | loss = nan, acc = 0.000
Epoch 1 | Step 1200 | loss = nan, acc = 0.000
Epoch 1 | Step 1300 | loss = nan, acc = 0.000
Epoch 1 | Step 1400 | loss = nan, acc = 0.000
Epoch 1 | Step 1500 | loss = nan, acc = 0.000
Epoch 1 | Step 1600 | loss = nan, acc = 0.000
Evaluating Dev Set ...


  0%|          | 0/3524 [00:00<?, ?it/s]

Validation | Epoch 1 | acc = 0.000
Saving Model ...


In [ ]:
### 10. Testing
print("Evaluating Test Set ...")

result = []

model.eval()
with torch.no_grad():
    for data in tqdm(test_loader):
        output = model(
            input_ids=data[0].squeeze(dim=0).to(device),
            token_type_ids=data[1].squeeze(dim=0).to(device),
            attention_mask=data[2].squeeze(dim=0).to(device),
        )
        result.append(evaluate(data, output))

result_file = "result.csv"
with open(result_file, "w") as f:
    f.write("ID,Answer\n")
    for i, test_question in enumerate(test_questions):
        # Replace commas in answers with empty strings (since csv is separated by comma)
        # (將答案中的逗號替換為空字串（因為 CSV 檔案是用逗號分隔的）。)
        # Answers in kaggle are processed in the same way
        f.write(f"{test_question['id']},{result[i].replace(',', '')}\n")

print(f"Completed! Result is in {result_file}")

Evaluating Test Set ...


  0%|          | 0/1000 [00:00<?, ?it/s]

Completed! Result is in result.csv
